# Chapter 3: Tensors and PyTorch

[Read this chapter online](https://jackluu.io/book/section-1-foundations/ch03-tensors-and-pytorch/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch03-tensors-and-pytorch.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 3: Tensors and PyTorch

![The map highlights the whole chain, as tensors are the foundation for everything](../assets/diagrams/ch03-where-we-are.png){ width="756" }
*Figure 3.1: Tensors are the containers that carry data through every stage.*

Now that we understand the language model's goal from the previous chapter, we are ready to start building (Figure 3.1). Before we turn text into numbers, we need a way to store and manipulate those numbers efficiently. Tensors are the specialized containers that do this job, and PyTorch is the engine that processes them. (Note: the exact numbers you see may differ slightly on your computer or PyTorch version).

In this chapter you will:

- Learn what a tensor is and how it stores data.
- See how tensor shapes represent different dimensions.
- Understand how matrix multiplication transforms data.
- Turn raw scores into probabilities using Softmax.

**Words to Know**
    - **PyTorch**: a library for doing math on large arrays of numbers very quickly.
    - **Tensor**: a multi-dimensional array of numbers.
    - **Shape**: the dimensions of a tensor (like rows and columns).
    - **Matrix Multiplication**: combining two tensors to transform data.
    - **Softmax**: a function that turns any numbers into probabilities that sum to 1.

## Theory: The Data Containers

PyTorch is a Python library for doing math on large arrays of numbers, really, really fast. Without PyTorch, training would take weeks instead of minutes because pure Python is too slow for millions of calculations.

![A 1D vector, a 2D matrix, and a 3D tensor shown as shapes](../assets/diagrams/ch03-tensor-shapes.png){ width="458" }
*Figure 3.2: Tensors can be 1D (a list), 2D (a table), or 3D (a stack of tables).*

A tensor is just a multi-dimensional array of numbers (Figure 3.2). If you have ever used a spreadsheet, you already know 2D tensors. A 3D tensor is just a stack of spreadsheets.

In this book, we work mostly with 3D tensors. Their dimensions, or **shape**, represent:

- **B** = Batch size (how many sequences we process at once)
- **T** = Time (how many tokens in each sequence)
- **C** = Channel size (how many numbers we use to represent each token)

We write shapes like `(B, T, C)`. For example, `(32, 128, 128)` means "32 sequences, each 128 tokens long, each token described by 128 numbers."

### Matrix Multiplication as Data Transformation

![A data tensor and a transformation tensor combine to make a result tensor](../assets/diagrams/ch03-matrix-multiply.png){ width="458" }
*Figure 3.3: Matrix multiplication transforms data from one shape to another.*

Matrix multiplication (written as `@` in Python) is how neural networks transform data, as shown in Figure 3.3. The rule is that the inner dimensions must match: a `(3, 4)` tensor can multiply a `(4, 5)` tensor, creating a new `(3, 5)` tensor. 

Think of `(3, 4)` as "3 students each with 4 test scores" and `(4, 5)` as "4 test scores each mapped to 5 skill ratings". Multiplying them gives you "3 students each with 5 skill ratings".

### Softmax: Turning Scores into Probabilities

As Figure 3.4 shows, Softmax is a mathematical function that takes any list of numbers (called logits) and turns them into probabilities. The probabilities are all positive and always sum to exactly 1.0. It does this using exponential functions, which make the highest numbers stand out even more.

![A diagram showing raw scores 1.0, 2.0, 3.0 turning into probabilities 0.09, 0.24, 0.67](../assets/diagrams/ch03-softmax.png){ width="468" }
*Figure 3.4: Softmax forces numbers into a 0-to-1 range where they total exactly 1.0.*

## Code: Basic Operations

Let's look at the basic PyTorch operations we will use.

```python
# A 1D tensor is simply a list of numbers
    a = torch.tensor([1.0, 2.0, 3.0, 4.0])

    # A 2D tensor represents a table or matrix of numbers
    b = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

    x = torch.randn(2, 5, 8)
    W = torch.randn(8, 4)
    y = x @ W
```

Run the file to see how PyTorch handles these:

```python
$ python src/ch02_tensors.py
--- 1. Creating tensors ---
1D tensor: tensor([1., 2., 3., 4.])
  shape: torch.Size([4])
...

...
        [ 0.2303, -1.1229, -0.1863]])
...
tensor([[ 0.,  1.,  2.,  3.],
...

...
```

**What just happened:**

1. Lines 2 and 5 created 1D and 2D tensors, printing their shapes.
2. Line 7 looked at a 3D tensor with a shape of `(2, 5, 8)`, representing `(Batch, Time, Channels)`.
3. Line 9 multiplied a 3D tensor by a 2D matrix. PyTorch automatically applied the multiplication across the batch dimension. This is called **batched matrix multiplication**, and it saves us from writing slow Python loops.

**Shape Check**

Table 3.1 shows the tensor shapes before and after matrix multiplication.

**Table 3.1:** Tensor shapes before and after matrix multiplication.

| Tensor | Shape | Meaning |
|---|---|---|
| `x` | `(2, 5, 8)` | 2 sequences, 5 tokens each, 8 numbers per token |
| `W` | `(8, 4)` | Transformation weights: 8 inputs to 4 outputs |
| `y` | `(2, 5, 4)` | 2 sequences, 5 tokens each, now 4 numbers per token |

**Try It**
    Open `src/ch02_tensors.py`, change `logits = torch.tensor([1.0, 2.0, 3.0])` to `[1.0, 2.0, 10.0]`, and run it. Notice how the highest number takes almost 100% of the probability after Softmax.

**In Business**
    How data is represented matters. When building our house-style email assistant, the text data is transformed into multi-dimensional tensors. The math operations we just covered are exactly how the assistant processes massive datasets to find hidden patterns in your company's writing voice.

**Watch Out**
    A shape mismatch is the most common error in PyTorch. If you try to multiply `(3, 4)` and `(5, 6)`, PyTorch will crash because the inner dimensions (4 and 5) do not match. Always check your shapes!

## Key Takeaways

- A tensor is a multi-dimensional array of numbers.
- Shape `(B, T, C)` = batch × time × channels, the standard convention in this book.
- `@` is matrix multiplication. Inner dimensions must match.
- Softmax turns any numbers into probabilities that sum to 1.
- PyTorch handles batches automatically using batched matrix multiplication, with no for-loops needed.
- With tensors ready to hold our data, we can move to the next stage in our map: turning text into tokens.

## Check Your Understanding
1. What does the shape `(32, 128, 128)` represent in our `(B, T, C)` format?
2. Why do the inner dimensions need to match in matrix multiplication?
3. What is the difference between logits and probabilities?


## Further Reading

**The library you are typing into.** The design argument behind the tool this book uses: write the model as ordinary Python that runs line by line, so you can print a tensor or stop in a debugger, and still get the speed of compiled code underneath. It is the reason the code in this book can be read top to bottom and still trains a real model.

<div class="refs" markdown>

Paszke, A., Gross, S., Massa, F., Lerer, A., Bradbury, J., Chanan, G., Killeen, T., Lin, Z., Gimelshein, N., Antiga, L., Desmaison, A., Köpf, A., Yang, E., DeVito, Z., Raison, M., Tejani, A., Chilamkurthy, S., Steiner, B., Fang, L., ... Chintala, S. (2019). *PyTorch: An imperative style, high-performance deep learning library* (arXiv:1912.01703). arXiv. https://doi.org/10.48550/arXiv.1912.01703

</div>

---

### `src/ch02_tensors.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch02_tensors.py"   # a cell has none, and the file uses it to find the text

"""
Introduce basic tensor operations and shapes.
This file belongs to Chapter 3.
Run: python src/ch02_tensors.py
"""
import torch

if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 3: Tensors and PyTorch Basics\n")

    print("--- 1. Creating tensors ---")
    # A 1D tensor is simply a list of numbers
    a = torch.tensor([1.0, 2.0, 3.0, 4.0])
    print(f"1D tensor: {a}\n  shape: {a.shape}")

    # A 2D tensor represents a table or matrix of numbers
    b = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    print(f"\n2D tensor:\n{b}\n  shape: {b.shape}")

    zeros = torch.zeros(3, 4)
    print(f"\nZeros (3x4):\n{zeros}")

    rand = torch.randn(2, 3)
    print(f"\nRandom (2x3):\n{rand}")

    print("\n--- 2. Shapes and dimensions ---")
    B, T, C = 2, 5, 8
    x = torch.randn(B, T, C)
    print(f"x shape: {x.shape}  - (batch={B}, time={T}, channels={C})")
    print(f"x[0] is the first sequence, shape: {x[0].shape}")
    print(
        f"x[0, 2] is the 3rd token of the 1st sequence, shape: {x[0, 2].shape}"
    )

    print("\n--- 3. Reshaping ---")
    flat = torch.arange(12, dtype=torch.float)
    print(f"Flat (12 numbers): {flat}")

    # Reshape changes the layout but keeps the total number of items the same
    grid = flat.reshape(3, 4)
    print(f"\nReshaped to (3, 4):\n{grid}")

    back_to_flat = grid.reshape(-1)
    print(f"\nBack to flat: {back_to_flat}")

    print("\n--- 4. Matrix multiplication ---")
    A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
    B_mat = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

    # The @ symbol performs matrix multiplication
    result = A @ B_mat
    print(f"A:\n{A}\nB:\n{B_mat}\nA @ B:\n{result}\n  shape: {result.shape}")

    x = torch.randn(2, 5, 8)
    W = torch.randn(8, 4)
    y = x @ W
    print(f"\n3D matmul: {x.shape} @ {W.shape} = {y.shape}")

    print("\n--- 5. Key operations used in transformers ---")
    logits = torch.tensor([1.0, 2.0, 3.0])

    # Softmax turns raw scores into probabilities that sum to 1
    probs = torch.softmax(logits, dim=0)
    probs_str = "[" + ", ".join(f"{p:.4f}" for p in probs) + "]"
    print(f"Softmax({logits.tolist()}) = {probs_str}")
    print(f"  Sum = {probs.sum():.4f}")

    mat = torch.randn(3, 5)
    
    # Transpose flips the axes, turning rows into columns and vice-versa
    mat_T = mat.transpose(-2, -1)
    print(f"\nTranspose: {mat.shape} -> {mat_T.shape}")

    print("\nReady for Chapter 4.")